# Chatbot RAG + Agente Jerárquico: Asistente de Investigación Académica — Evaluación 2

**Caso organizacional:** un investigador recibe papers en otro idioma. El agente los traduce, los resume usando RAG y genera un PDF de ficha de lectura. Además, ahora el agente **recuerda la conversación** (memoria de corto plazo), **planifica qué estrategia seguir** antes de actuar (IL2.3), y expone sus funciones como **herramientas** dentro de un framework de agentes (CrewAI, IL2.1).

**Novedades respecto a la Evaluación 1:**
- 🧠 **Memoria de corto plazo**: el bot ahora usa el historial de la conversación (Gradio ya lo entrega, antes se ignoraba).
- 📚 **Memoria de largo plazo / recuperación semántica**: el índice FAISS con embeddings de Gemini (ya existía, se documenta como tal).
- 🧭 **Planificador de estrategia**: antes de actuar, un agente decide QUÉ hacer según el mensaje, si trae PDF, y el historial — y lo explica (toma de decisiones adaptativa).
- 🏛️ **Capa de agentes jerárquica con CrewAI**: Manager + Agentes especializados (Investigador, Redactor, Conversacional) que reutilizan las mismas funciones como *herramientas*, demostrando el uso de un framework específico de agentes.

**Tecnologías:** Groq (LLM) · Gemini (embeddings) · FAISS (vectorial) · CrewAI (orquestación jerárquica) · Gradio (interfaz)

> Ejecuta las celdas en orden de arriba hacia abajo.

## Celda 1: Instalación de librerías

In [ ]:
# INSTALACION DE LIBRERIAS
# Se agregan respecto a la Evaluacion 1:
#   crewai        -> framework de orquestacion de agentes (IL2.1): Manager + agentes especializados
#   pydantic      -> esquemas de entrada para las herramientas de CrewAI
import subprocess
subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "openai", "google-generativeai", "faiss-cpu",
    "pypdf", "reportlab", "gradio>=5.0", "langchain-text-splitters",
    "crewai", "pydantic"
])

## Celda 2: Configuración de claves y clientes

In [ ]:
# CONFIGURACION DE CLAVES Y CLIENTES
# El chatbot necesita dos API Keys guardadas como Secrets en Colab
# (icono de llave en la barra lateral izquierda):
#
#   GROQ_API_KEY   -> obtenla en console.groq.com/keys
#   GEMINI_API_KEY -> obtenla en aistudio.google.com/apikey

import os
import time
from google.colab import userdata
from openai import OpenAI, RateLimitError, NotFoundError
import google.generativeai as genai

# -- Cliente de chat (Groq) ------------------------------------------
# Groq es compatible con el formato de OpenAI, por eso usamos el mismo
# cliente pero apuntando a la URL de Groq.
LLM_API_KEY  = userdata.get("GROQ_API_KEY")
LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL    = "openai/gpt-oss-120b"      # modelo con buenos limites en free tier
LLM_MODEL_FALLBACK = "qwen/qwen3.8-27b"   # si el modelo principal no esta disponible

client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

print("Base URL configurada:", LLM_BASE_URL)
print("Groq API Key configurada:", "OK" if LLM_API_KEY else "FALTA el secreto GROQ_API_KEY")

# -- Cliente de embeddings (Gemini) ----------------------------------
# Gemini se usa SOLO para generar embeddings (memoria de largo plazo / RAG).
GEMINI_API_KEY  = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
EMBEDDING_MODEL = "models/gemini-embedding-001"

print("Gemini API Key configurada:", "OK" if GEMINI_API_KEY else "FALTA el secreto GEMINI_API_KEY")


def llamar_llm(mensajes, temperature=0.2, max_tokens=1200, modelo=None):
    """
    Funcion base que llama a Groq, con reintento automatico:
      - Si el modelo principal no existe (404), cambia al modelo de respaldo.
      - Si hay rate limit (429), espera y reintenta una vez.
    """
    modelo = modelo or LLM_MODEL
    try:
        respuesta = client.chat.completions.create(
            model=modelo, messages=mensajes, temperature=temperature, max_tokens=max_tokens,
        )
        return respuesta.choices[0].message.content
    except NotFoundError:
        if modelo != LLM_MODEL_FALLBACK:
            print(f"Modelo '{modelo}' no disponible, usando respaldo '{LLM_MODEL_FALLBACK}'")
            return llamar_llm(mensajes, temperature, max_tokens, modelo=LLM_MODEL_FALLBACK)
        raise
    except RateLimitError:
        print("Rate limit alcanzado, esperando 20s y reintentando...")
        time.sleep(20)
        respuesta = client.chat.completions.create(
            model=modelo, messages=mensajes, temperature=temperature, max_tokens=max_tokens,
        )
        return respuesta.choices[0].message.content

print("Funcion llamar_llm (con reintento) definida correctamente")

## Celda 3: Base de conocimiento (memoria de largo plazo / RAG)

In [ ]:
# BASE DE CONOCIMIENTO (RAG = memoria de largo plazo semantica)
# RAG = Retrieval-Augmented Generation.
# En vez de que el modelo responda solo desde su entrenamiento,
# primero RECUPERAMOS informacion relevante de nuestra propia base
# de datos y se la entregamos como contexto.
#
#   tipo='faq'   -> info general sobre el chatbot (modo conversacion).
#   tipo='paper' -> fragmentos de papers relacionados (modo tarea).

faq_docs = [
    "Soy un asistente de investigacion academica. Puedo traducir papers, resumirlos y generar una ficha de lectura en PDF.",
    "Para usarme, simplemente sube un archivo PDF de un paper en el chat y yo me encargo del resto.",
    "Puedo conversar normalmente si solo quieres saludar o hacer una pregunta general.",
    "Tengo memoria de la conversacion actual: recuerdo lo que hemos hablado en este chat.",
    "Antes de actuar, un agente Planificador decide la mejor estrategia segun tu mensaje y el historial.",
    "Fui construido con Groq como modelo de lenguaje, Gemini para los embeddings, y CrewAI para orquestar agentes especializados.",
]

# TODO: reemplaza estos textos con fragmentos reales de papers de tu area
paper_context_docs = [
    "Los modelos LLM basados en transformers han demostrado mejoras en tareas de resumen automatico.",
    "La tecnica RAG combina recuperacion de informacion con generacion de texto, reduciendo alucinaciones.",
    "La calidad de la traduccion automatica depende fuertemente del contexto de dominio proporcionado.",
]

knowledge_base = ([{"texto": t, "tipo": "faq"} for t in faq_docs] +
                  [{"texto": t, "tipo": "paper"} for t in paper_context_docs])

print(f"Base de conocimiento: {len(knowledge_base)} documentos ({len(faq_docs)} FAQ + {len(paper_context_docs)} papers)")

## Celda 4: Construcción del índice vectorial (FAISS)

In [ ]:
# INDICE VECTORIAL CON FAISS (memoria de largo plazo / recuperacion semantica)
import numpy as np
import faiss

def obtener_embedding(texto):
    resultado = genai.embed_content(model=EMBEDDING_MODEL, content=texto)
    return np.array(resultado["embedding"], dtype="float32")

def construir_indice(kb):
    print("Generando embeddings...")
    vectores = np.array([obtener_embedding(d["texto"]) for d in kb], dtype="float32")
    dimension = vectores.shape[1]
    indice = faiss.IndexFlatL2(dimension)
    indice.add(vectores)
    return indice

def buscar_contexto(pregunta, kb, indice, k=3):
    vector_pregunta = obtener_embedding(pregunta).reshape(1, -1)
    _, posiciones = indice.search(vector_pregunta, k)
    return [kb[i] for i in posiciones[0] if i < len(kb)]

indice_faiss = construir_indice(knowledge_base)
print(f"Indice FAISS listo con {indice_faiss.ntotal} vectores indexados")

## Celda 5: Herramientas del agente (consulta, traducción, resumen)

Estas funciones son las **herramientas** (tools) que el agente usa para cumplir tareas específicas — IL2.1 pide explícitamente "herramientas de consulta, escritura y razonamiento". Aquí están etiquetadas según su rol:

- `herramienta_lectura_pdf` → **consulta** (lee el PDF de entrada)
- `herramienta_traduccion` → **escritura** (produce texto traducido)
- `herramienta_resumen_rag` → **razonamiento** (combina contexto RAG + LLM para resumir)

In [ ]:
from pypdf import PdfReader

def herramienta_lectura_pdf(ruta_pdf):
    """Herramienta de CONSULTA: extrae el texto de un PDF."""
    lector = PdfReader(ruta_pdf)
    paginas = [pagina.extract_text() or "" for pagina in lector.pages]
    return chr(10).join(paginas)

def herramienta_traduccion(texto, idioma_destino='espanol'):
    """Herramienta de ESCRITURA: traduce un texto al idioma destino."""
    mensajes = [
        {"role": "system", "content": f"Eres un traductor experto. Traduce el texto al {idioma_destino} manteniendo el sentido tecnico. Responde solo con la traduccion."},
        {"role": "user", "content": texto[:8000]},
    ]
    return llamar_llm(mensajes, temperature=0.1)

def herramienta_resumen_rag(texto_traducido, kb, indice):
    """Herramienta de RAZONAMIENTO: resume un texto usando contexto RAG recuperado de FAISS."""
    contexto   = buscar_contexto(texto_traducido[:500], kb, indice, k=3)
    ctx_papers = [c["texto"] for c in contexto if c["tipo"] == "paper"]
    sep = chr(10)
    ctx_texto  = sep.join(f"- {c}" for c in ctx_papers) or "Sin contexto adicional disponible."
    mensajes = [
        {"role": "system", "content": (
            "Eres un asistente de investigacion academica. "
            "Resume el paper de forma clara, usando el contexto como apoyo, "
            "sin inventar datos ni citar fuentes no entregadas."
        )},
        {"role": "user", "content": (
            "CONTEXTO DE PAPERS RELACIONADOS:" + chr(10) + ctx_texto + chr(10)*2 +
            "PAPER A RESUMIR:" + chr(10) + texto_traducido[:6000] + chr(10)*2 +
            "Genera: 1) resumen de 200-300 palabras. 2) 3-5 puntos clave con guion."
        )},
    ]
    resumen = llamar_llm(mensajes, temperature=0.2)
    return resumen, ctx_papers

print("Herramientas del agente (consulta, escritura, razonamiento) definidas correctamente")

## Celda 6: Generador de PDF (ficha de lectura) — herramienta de escritura final

In [ ]:
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
from reportlab.lib.utils import simpleSplit

import re

def limpiar_texto_pdf(texto):
    """Quita markdown y reemplaza caracteres que Helvetica no dibuja (evita cuadros negros)."""
    texto = str(texto)
    texto = re.sub(r"\*\*|__|`", "", texto)                 # negritas / codigo markdown
    texto = re.sub(r"^#+\s*", "", texto, flags=re.M)          # encabezados markdown
    texto = re.sub(r"^\s*[\*\-]\s+", "- ", texto, flags=re.M)  # viñetas markdown
    for malo in ("\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212"):
        texto = texto.replace(malo, "-")                      # guiones unicode -> '-'
    for esp in ("\u00a0", "\u202f", "\u2009", "\u200b"):
        texto = texto.replace(esp, " ")                       # espacios raros
    return texto

def herramienta_generar_pdf(titulo, resumen, citas, ruta_salida="/content/ficha_lectura.pdf"):
    """Herramienta de ESCRITURA: genera el PDF de ficha de lectura final."""
    titulo  = limpiar_texto_pdf(titulo)
    resumen = limpiar_texto_pdf(resumen)
    citas   = [limpiar_texto_pdf(x) for x in citas]
    c = canvas.Canvas(ruta_salida, pagesize=LETTER)
    ancho, alto = LETTER
    margen = 2 * cm
    y = alto - margen

    c.setFont("Helvetica-Bold", 16)
    c.drawString(margen, y, "Ficha de Lectura")
    y -= 1 * cm

    c.setFont("Helvetica-Bold", 12)
    for linea in simpleSplit(titulo, "Helvetica-Bold", 12, ancho - 2*margen):   # titulo en varias lineas
        c.drawString(margen, y, linea)
        y -= 0.55 * cm
    y -= 0.45 * cm

    c.setFont("Helvetica-Bold", 11)
    c.drawString(margen, y, "Resumen:")
    y -= 0.6 * cm
    c.setFont("Helvetica", 10)
    for linea in simpleSplit(resumen, "Helvetica", 10, ancho - 2*margen):
        if y < margen:
            c.showPage()
            y = alto - margen
        c.drawString(margen, y, linea)
        y -= 0.5 * cm

    y -= 0.5 * cm
    c.setFont("Helvetica-Bold", 11)
    c.drawString(margen, y, "Contexto / citas recuperadas por RAG:")
    y -= 0.6 * cm
    c.setFont("Helvetica-Oblique", 9)
    for cita in citas:
        for linea in simpleSplit(f"- {cita}", "Helvetica-Oblique", 9, ancho - 2*margen):
            if y < margen:
                c.showPage()
                y = alto - margen
            c.drawString(margen, y, linea)
            y -= 0.45 * cm

    c.save()
    print(f"PDF generado en: {ruta_salida}")
    return ruta_salida

print("Herramienta generadora de PDF definida correctamente")

## Celda 7: Memoria de corto plazo (IL2.2)

Gradio ya entrega el historial de la conversación en cada llamada a `responder_chat(mensaje, historial)` — en la Evaluación 1 ese parámetro se recibía pero **nunca se usaba**. Aquí lo convertimos en **memoria de corto plazo real**: los últimos turnos se incluyen como contexto en cada llamada al LLM, para que el bot recuerde lo que se habló antes en la misma sesión.

In [ ]:
MAX_TURNOS_MEMORIA = 6  # cuantos turnos previos recordar (ventana de memoria de corto plazo)

def construir_mensajes_memoria(historial):
    """
    Convierte el historial de Gradio (lista de turnos previos) en una lista
    de mensajes 'user'/'assistant' que se puede inyectar directamente en la
    llamada al LLM. Esto es la memoria de CORTO PLAZO: solo dura la sesion
    actual del chat, a diferencia del FAISS que es memoria de LARGO PLAZO
    (persiste entre sesiones porque vive en la base de conocimiento).
    """
    if not historial:
        return []

    turnos_recientes = historial[-MAX_TURNOS_MEMORIA:]
    mensajes_memoria = []
    for turno in turnos_recientes:
        # Gradio (type='messages') entrega cada turno como dict {'role':..., 'content':...}
        # o como tupla (usuario, bot) en versiones antiguas. Soportamos ambos formatos.
        if isinstance(turno, dict):
            rol = turno.get("role")
            contenido = turno.get("content", "")
            if isinstance(contenido, dict):  # mensaje multimodal (con archivos)
                contenido = contenido.get("text", "")
            elif isinstance(contenido, list):  # Gradio 5+: lista de partes [{'text':..., 'type':'text'}, {'path':...}]
                partes = []
                for parte in contenido:
                    if isinstance(parte, dict):
                        if parte.get("type") == "text" or "text" in parte:
                            partes.append(str(parte.get("text", "")))
                    elif isinstance(parte, str):
                        partes.append(parte)
                contenido = " ".join(t for t in partes if t).strip()
            if rol in ("user", "assistant") and contenido:
                mensajes_memoria.append({"role": rol, "content": str(contenido)[:500]})
        elif isinstance(turno, (list, tuple)) and len(turno) == 2:
            usuario, bot = turno
            if usuario:
                mensajes_memoria.append({"role": "user", "content": str(usuario)[:500]})
            if bot:
                mensajes_memoria.append({"role": "assistant", "content": str(bot)[:500]})

    return mensajes_memoria

print("Memoria de corto plazo (historial conversacional) definida correctamente")

## Celda 8: Agente Planificador — toma de decisiones adaptativa (IL2.3)

Antes de ejecutar cualquier acción, un **agente Planificador** analiza el mensaje, si trae un PDF adjunto, y la memoria de corto plazo, y decide qué **estrategia** seguir — explicando por qué. Esto demuestra que el comportamiento del sistema se ajusta ante condiciones cambiantes (no sigue siempre el mismo camino fijo).

In [ ]:
import json

ESTRATEGIAS_DISPONIBLES = {
    "PROCESAR_PAPER": "Hay un PDF adjunto: traducir, resumir con RAG y generar ficha de lectura.",
    "RESPUESTA_CON_RAG": "Pregunta sobre el propio sistema o tema cubierto por la base de conocimiento: usar FAQ/RAG.",
    "RESPUESTA_CON_MEMORIA": "Pregunta de seguimiento que depende de turnos anteriores: usar memoria de corto plazo.",
    "CONVERSACION_LIBRE": "Saludo o charla general sin necesidad de contexto adicional.",
}

def planificar_estrategia(texto_usuario, hay_archivo, mensajes_memoria):
    """
    Agente Planificador (IL2.3): decide la estrategia a seguir antes de actuar.
    No ejecuta nada por si mismo, solo RAZONA y devuelve una decision explicable,
    igual que el patron Plan-and-Execute visto en la unidad 2.3.
    """
    # Regla dura: si hay archivo, la estrategia siempre es procesar el paper.
    # (una decision de negocio clara no necesita gastar una llamada al LLM)
    if hay_archivo:
        return {"estrategia": "PROCESAR_PAPER", "razonamiento": "El mensaje trae un archivo PDF adjunto."}

    hay_memoria_previa = len(mensajes_memoria) > 0

    prompt_planificador = (
        "Eres un agente planificador. Dado el mensaje del usuario, decide UNA de estas estrategias:\n"
        + chr(10).join(f"- {k}: {v}" for k, v in ESTRATEGIAS_DISPONIBLES.items() if k != "PROCESAR_PAPER")
        + "\n\nResponde SOLO en JSON con este formato exacto: "
        + '{"estrategia": "NOMBRE_ESTRATEGIA", "razonamiento": "explicacion breve de 1 linea"}'
    )
    mensajes = [{"role": "system", "content": prompt_planificador}]
    if hay_memoria_previa:
        mensajes.append({"role": "system", "content": f"Hay {len(mensajes_memoria)} mensajes previos en esta conversacion."})
    mensajes.append({"role": "user", "content": texto_usuario})

    try:
        respuesta = llamar_llm(mensajes, temperature=0.0, max_tokens=150)
        inicio, fin = respuesta.find("{"), respuesta.rfind("}") + 1
        decision = json.loads(respuesta[inicio:fin])
        if decision.get("estrategia") not in ESTRATEGIAS_DISPONIBLES:
            raise ValueError("estrategia fuera de catalogo")
        return decision
    except Exception:
        # Fallback seguro si el LLM no devuelve un JSON valido: conversacion libre
        return {"estrategia": "CONVERSACION_LIBRE", "razonamiento": "No se pudo determinar una estrategia especifica, se usa conversacion general."}

print("Agente Planificador definido correctamente")

## Celda 9: Orquestador del chatbot (usa memoria + planificador)

Flujo actualizado:

```
Mensaje + historial
     |
     v
Agente Planificador  --> decide estrategia (y la muestra en la respuesta)
     |
     +-- PROCESAR_PAPER        -> herramienta_lectura_pdf + traduccion + resumen RAG + PDF
     +-- RESPUESTA_CON_RAG     -> busca en FAISS (FAQ) + responde
     +-- RESPUESTA_CON_MEMORIA -> incluye memoria de corto plazo + responde
     +-- CONVERSACION_LIBRE    -> respuesta directa con memoria de corto plazo
```

In [ ]:
def responder_chat(mensaje, historial):
    archivos      = mensaje.get("files", []) if isinstance(mensaje, dict) else []
    texto_usuario = mensaje.get("text", "") if isinstance(mensaje, dict) else mensaje

    mensajes_memoria = construir_mensajes_memoria(historial)

    # --- Paso 1: el Planificador decide la estrategia ---------------
    decision = planificar_estrategia(texto_usuario, bool(archivos), mensajes_memoria)
    estrategia = decision["estrategia"]
    print(f"[Planificador] Estrategia elegida: {estrategia} -> {decision['razonamiento']}")

    # --- Paso 2: ejecutar segun la estrategia ------------------------
    # Solo el modo PROCESAR_PAPER devuelve un dict (porque adjunta un PDF).
    # Los demas modos devuelven un string plano, igual que en la Evaluacion 1,
    # para que Gradio los renderice como texto normal en la burbuja del chat.
    if estrategia == "PROCESAR_PAPER":
        ruta_pdf = archivos[0]
        texto_original = herramienta_lectura_pdf(ruta_pdf)
        if not texto_original.strip():
            return "No pude leer texto del PDF. Verifica que no sea una imagen escaneada sin OCR."

        texto_traducido = herramienta_traduccion(texto_original)
        resumen, citas = herramienta_resumen_rag(texto_traducido, knowledge_base, indice_faiss)
        titulo = texto_traducido.split(chr(10))[0][:90] or 'Paper sin titulo'
        ruta_pdf_final = herramienta_generar_pdf(titulo, resumen, citas)

        return {
            "text":  f"[Estrategia: {estrategia}] Listo, procese tu paper. Aqui tienes el resumen:" + chr(10)*2 + resumen,
            "files": [ruta_pdf_final],
        }

    elif estrategia == "RESPUESTA_CON_RAG":
        contexto      = buscar_contexto(texto_usuario, knowledge_base, indice_faiss, k=2)
        contexto_faq  = [c["texto"] for c in contexto if c["tipo"] == "faq"]
        contexto_txt  = chr(10).join(contexto_faq)
        mensajes = [{"role": "system", "content": (
            "Eres un asistente amable de investigacion academica. "
            "Responde de forma breve, clara y natural. Informacion sobre ti mismo:" + chr(10) + contexto_txt
        )}] + mensajes_memoria + [{"role": "user", "content": texto_usuario}]
        return llamar_llm(mensajes, temperature=0.4, max_tokens=300)

    elif estrategia == "RESPUESTA_CON_MEMORIA":
        mensajes = [{"role": "system", "content": (
            "Eres un asistente de investigacion academica con memoria de la conversacion actual. "
            "Usa los mensajes previos para responder de forma coherente y con continuidad."
        )}] + mensajes_memoria + [{"role": "user", "content": texto_usuario}]
        return llamar_llm(mensajes, temperature=0.3, max_tokens=400)

    else:  # CONVERSACION_LIBRE
        mensajes = [{"role": "system", "content": "Eres un asistente amable. Responde de forma breve y natural."}] \
            + mensajes_memoria + [{"role": "user", "content": texto_usuario}]
        return llamar_llm(mensajes, temperature=0.5, max_tokens=250)

print("Orquestador del chatbot (con memoria + planificador) definido correctamente")

## Celda 10 (evidencia IL2.1): Capa de agentes jerárquica con CrewAI

Esta celda demuestra el **framework específico de agentes** (CrewAI) requerido por IL2.1, reutilizando exactamente las mismas herramientas (`herramienta_traduccion`, `herramienta_resumen_rag`, `herramienta_generar_pdf`) ya definidas, envueltas como `BaseTool`. Un **Manager jerárquico** decide a qué agente delegar — igual que en el notebook `4-multiagent_crewai_hierarchical.py` de la unidad 2.3. Esta celda es independiente del chat de Gradio: se usa como evidencia técnica adicional en el repositorio/demo de que el sistema SÍ puede operar bajo un framework de agentes formal, no solo con funciones sueltas.

In [ ]:
from typing import Type
from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

_crewai_llm_base = dict(
    model="openai/" + LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0.3,
)
llm_crewai_worker  = LLM(**_crewai_llm_base, max_tokens=700)
llm_crewai_manager = LLM(**_crewai_llm_base, max_tokens=500)


class TraduccionInput(BaseModel):
    texto: str = Field(description="Texto en ingles a traducir al espanol.")

class HerramientaTraduccionCrewAI(BaseTool):
    name: str = "traducir_texto"
    description: str = "Traduce un texto al espanol manteniendo el sentido tecnico."
    args_schema: Type[BaseModel] = TraduccionInput

    def _run(self, texto: str) -> str:
        return herramienta_traduccion(texto)


class ResumenInput(BaseModel):
    texto_traducido: str = Field(description="Texto ya traducido al espanol que se debe resumir.")

class HerramientaResumenCrewAI(BaseTool):
    name: str = "resumir_con_rag"
    description: str = "Resume un texto en espanol usando contexto RAG de la base de conocimiento de papers."
    args_schema: Type[BaseModel] = ResumenInput

    def _run(self, texto_traducido: str) -> str:
        resumen, citas = herramienta_resumen_rag(texto_traducido, knowledge_base, indice_faiss)
        return resumen + chr(10)*2 + "Citas: " + "; ".join(citas)


herramienta_traduccion_tool = HerramientaTraduccionCrewAI()
herramienta_resumen_tool    = HerramientaResumenCrewAI()

agente_investigador = Agent(
    role="Investigador Academico",
    goal="Traducir papers en ingles al espanol manteniendo el sentido tecnico.",
    backstory="Eres un traductor academico especializado en papers cientificos.",
    llm=llm_crewai_worker,
    tools=[herramienta_traduccion_tool],
    max_iter=3,
    verbose=True,
)

agente_analista = Agent(
    role="Analista de Investigacion",
    goal="Resumir papers traducidos usando contexto RAG, identificando puntos clave.",
    backstory="Eres un analista experto en sintetizar papers academicos en resumenes claros.",
    llm=llm_crewai_worker,
    tools=[herramienta_resumen_tool],
    max_iter=3,
    verbose=True,
)

tarea_traducir = Task(
    description="Traduce el siguiente texto de un paper al espanol: {texto_paper}",
    expected_output="El texto completo traducido al espanol, manteniendo terminologia tecnica.",
    agent=agente_investigador,
)

tarea_resumir = Task(
    description="Resume el texto traducido en la tarea anterior, usando contexto RAG. Entrega resumen y puntos clave.",
    expected_output="Resumen de 200-300 palabras con puntos clave.",
    agent=agente_analista,
)

crew_jerarquico = Crew(
    agents=[agente_investigador, agente_analista],
    tasks=[tarea_traducir, tarea_resumir],
    process=Process.hierarchical,
    manager_llm=llm_crewai_manager,
    verbose=True,
)

print("Crew jerarquico (Manager + Investigador + Analista) definido correctamente.")
print("Para probarlo: crew_jerarquico.kickoff(inputs={'texto_paper': 'texto en ingles aqui...'})")

## Celda 11: Interfaz de chat con Gradio

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=responder_chat,
    multimodal=True,
    title="Asistente de Investigacion Academica (con memoria y planificacion)",
    description=(
        "Saluda o preguntame algo, pide que recuerde algo de la conversacion, "
        "o sube un paper en PDF y lo traduzco, resumo y genero una ficha de lectura. "
        "Un agente Planificador decide la estrategia antes de responder (se ve en la consola)."
    ),
    examples=[
        {"text": "Hola, que puedes hacer?", "files": []},
        {"text": "Como funciona el RAG?", "files": []},
    ],
)

demo.launch(debug=True, share=True)